In [2]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time
import os
from pathlib import Path
import datetime

In [12]:
current_directory = os.getcwd()
csv_name = os.path.join(current_directory, "..", "data", "prod_urls.csv")
prod_url = pd.read_csv(csv_name)
prod_url.head(5)

,product_id,url,status,discovered_at
0,698859,https://www.naiin.com/product/detail/698859/,pending,2026-03-15 16:23:55.176126
1,698857,https://www.naiin.com/product/detail/698857/,pending,2026-03-15 16:23:55.176126
2,698856,https://www.naiin.com/product/detail/698856/,pending,2026-03-15 16:23:55.176126
3,699421,https://www.naiin.com/product/detail/699421/,pending,2026-03-15 16:23:55.176126
4,698858,https://www.naiin.com/product/detail/698858/,pending,2026-03-15 16:23:55.176126


In [13]:
# 1. กำหนดลิสต์ ID ที่ต้องการทดสอบ
target_ids = [542979, 691950, 697743, 542641, 697305]

# 2. กรองข้อมูลจาก DataFrame หลัก (prod_url)
examples = prod_url[prod_url['product_id'].isin(target_ids)]

In [3]:
prod_url = prod_url.drop(columns=['status', 'discovered_at'])
prod_url['product_id'] = prod_url['product_id'].astype(int)
prod_url.head(5)

,product_id,url
0,698859,https://www.naiin.com/product/detail/698859/
1,698857,https://www.naiin.com/product/detail/698857/
2,698856,https://www.naiin.com/product/detail/698856/
3,699421,https://www.naiin.com/product/detail/699421/
4,698858,https://www.naiin.com/product/detail/698858/


In [4]:
examples = prod_url.head(5)
examples

,product_id,url
0,698859,https://www.naiin.com/product/detail/698859/
1,698857,https://www.naiin.com/product/detail/698857/
2,698856,https://www.naiin.com/product/detail/698856/
3,699421,https://www.naiin.com/product/detail/699421/
4,698858,https://www.naiin.com/product/detail/698858/


In [3]:
import re
import json
import html

def extract_extra_info(html_content):
    extra_data = {}
    # ใช้ BeautifulSoup ช่วยสำหรับส่วนที่ JSON ไม่มีข้อมูล
    soup = BeautifulSoup(html_content, 'lxml')
    
    try:
        # 1. ดึง Rating และข้อมูลเบื้องต้นจาก JSON (วิธีเดิมที่คุณใช้แล้วได้ผล)
        match = re.search(r'AverageRating&quot;:(.+?)\}', html_content)
        if match:
            raw_json = '{"AverageRating":' + match.group(1) + '}'
            clean_json = html.unescape(raw_json)
            json_obj = json.loads(clean_json)
            
            extra_data = {
                'AverageRating': json_obj.get('AverageRating') or None,
                'TotalRating': json_obj.get('TotalRating') or None,
                'NumberOfPage': json_obj.get('NumberOfPage') or None,
                # เริ่มต้นด้วย None เดี๋ยวเราใช้ BeautifulSoup ทับถ้าหาเจอ
                'Width': None,
                'Height': None,
                'Thick': None,
                'GrossWeightKG': None,
                'FileSizeMB': None
            }

        # 2. ใช้ BeautifulSoup เจาะหา "ขนาดไฟล์", "ขนาด", "น้ำหนัก" จาก Label โดยตรง
        # เราจะหา <label> ที่มีข้อความที่ต้องการ แล้วหา <p> ที่อยู่ถัดจากมัน
        labels = soup.find_all('label', class_='product-label')
        for label in labels:
            text = label.get_text(strip=True)
            detail_p = label.find_next_sibling('p', class_='product-label-detail')
            
            if detail_p:
                val = detail_p.get_text(strip=True)
                
                if "ขนาดไฟล์" in text:
                    # ดึงเฉพาะตัวเลขจาก "4.08 MB"
                    size_val = re.search(r'[\d\.]+', val)
                    if size_val: extra_data['FileSizeMB'] = size_val.group()
                
                elif "ขนาด" in text:
                    # แยก "0 x 0 x 0 CM" ออกเป็น 3 ส่วน
                    dims = re.findall(r'[\d\.]+', val)
                    if len(dims) >= 3:
                        extra_data['Width'] = dims[0]
                        extra_data['Height'] = dims[1]
                        extra_data['Thick'] = dims[2]
                
                elif "น้ำหนัก" in text:
                    # ดึงเฉพาะตัวเลขจาก "0.374 KG"
                    weight_val = re.search(r'[\d\.]+', val)
                    if weight_val: extra_data['GrossWeightKG'] = weight_val.group()

    except Exception as e:
        print(f"  [Warning] Extra info skip: {e}")
        
    return extra_data

In [ ]:
def scrape(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.naiin.com/',
    }
    
    # เตรียมข้อมูลตั้งต้น
    data = {
        'url': url,
        'status_code': None,
        'Title': "N/A",
        'Price_Full': "N/A",
        'Barcode': "N/A",
        'Release_Date': "N/A",
        'keywords': "N/A",
        'product_type': "N/A",
        'author': "N/A",
        'publisher': "N/A",
        'category_lv1': "N/A",
        'category_lv2': "N/A"
    }
    
    try:
        time.sleep(1.5)
        response = requests.get(url, headers=headers, timeout=10)
        data['status_code'] = response.status_code
        
        # ถ้า status ไม่ใช่ 200 (เช่น 404) ให้คืนค่าเลย ไม่ต้องเสียเวลา parse
        if response.status_code != 200:
            return data

        html_text = response.text
        soup = BeautifulSoup(html_text, 'lxml')
        
        # 1. ดึง Metadata ปกติ
        meta_map = {
            'Title': soup.find('meta', property='og:title'),
            'Price_Full': soup.find('meta', property='og:product:price:amount'),
            'Barcode': soup.find('meta', property='book:isbn'),
            'Release_Date': soup.find('meta', property='book:release_date'),
        }
        
        for key, tag in meta_map.items():
            if tag and tag.has_attr('content'):
                data[key] = tag['content'].strip()

        # 2. ดึง Keywords ด้วย Greedy Regex
        kw_pattern = r'<meta[^>]*name="keywords"[^>]*content="(.*)"\s*/?>'
        kw_match = re.search(kw_pattern, html_text, re.IGNORECASE | re.DOTALL)

        if kw_match:
            raw_content = kw_match.group(1).strip()
            clean_content = raw_content.split('">')[0]
            data['keywords'] = clean_content.rstrip('"')

        # 3. ดึง Extra Info (Table/Rating)
        extra_info = extract_extra_info(html_text)
        data.update(extra_info)

        # 4. ดึง Product Type จาก Breadcrumbs ---
        data['product_type'] = "N/A" # ค่าตั้งต้น
        breadcrumb_div = soup.find('div', class_='breadcrumbs')

        if breadcrumb_div:
            # หา <a> ทั้งหมดใน breadcrumbs
            links = breadcrumb_div.find_all('a')
            # ปกติ links[0] คือ "หน้าแรก", links[1] คือ "Product Type"
            if len(links) >= 2:
                data['product_type'] = links[1].get_text(strip=True)
        
        # 5. ดึงข้อมูล ผู้เขียน, สำนักพิมพ์, หมวดหมู่ (ส่วนที่เพิ่มใหม่)
        container = soup.find('div', class_='bookdetail-container')
        if container:
            labels = container.find_all('label', class_='product-label')
            for label in labels:
                text = label.get_text(strip=True)
                # หาแท็ก <a> ที่อยู่ถัดจาก label
                value_tag = label.find_next_sibling('a', class_='link-book-detail')
                
                if value_tag:
                    value = value_tag.get_text(strip=True)
                    if "ผู้เขียน:" in text:
                        data['author'] = value
                    elif "สำนักพิมพ์:" in text:
                        data['publisher'] = value
                    elif "หมวดหมู่:" in text:
                        data['category_lv1'] = value
                        # หาหมวดหมู่ย่อย (ถ้ามี) ซึ่งจะเป็น <a> ตัวถัดไป
                        lv2_tag = value_tag.find_next_sibling('a', class_='link-book-detail')
                        if lv2_tag:
                            data['category_lv2'] = lv2_tag.get_text(strip=True)

    except Exception as e:
        print(f"  [Request Error] {url}: {e}")
        data['status_code'] = 999 # กำหนดรหัสพิเศษสำหรับ Network Error
        data['Title'] = None
        
    return data

In [11]:
scrape('https://www.naiin.com/product/detail/664433')

{'url': 'https://www.naiin.com/product/detail/664433',
 'status_code': 200,
 'Title': 'แพรว ฉ.1021 (ส.ค.68 อิงฟ้า วราหะ)Magazines | ร้านหนังสือนายอินทร์',
 'Price_Full': '140',
 'Barcode': '977012568581908',
 'Release_Date': '2025-08-04',
 'keywords': 'แพรว ฉ.1021 (ส.ค.68 อิงฟ้า วราหะ), , นิตยสารแพรว, Magazine, นิตยสารในเครืออมรินทร์, ',
 'product_type': 'นิตยสาร',
 'author': 'N/A',
 'publisher': 'นิตยสารแพรว',
 'category_lv1': 'Magazine',
 'category_lv2': 'นิตยสารในเครืออมรินทร์',
 'AverageRating': 5,
 'TotalRating': 2,
 'NumberOfPage': 192,
 'Width': '0',
 'Height': '0',
 'Thick': '0',
 'GrossWeightKG': '0.621',
 'FileSizeMB': None}

In [14]:

test_results = []
for _, row in examples.iterrows():
    p_id = row['product_id']
    url = row['url']
    
    try:
        # เรียกใช้ฟังก์ชัน scrape ที่เราเพิ่งแก้
        res = scrape(url) 
        
        # จำลองการสร้าง row แบบที่จะลง DB
        final_row = {
            'product_id': p_id,
            **res,
            'scraped_at': datetime.datetime.now()
        }
        test_results.append(final_row)
        print(f"✅ Scraped ID {p_id}: {res['Title'][:30]}...")
        
    except Exception as e:
        print(f"❌ Error ID {p_id}: {e}")

✅ Scraped ID 542979: INPUT - OUTPUT สุดยอดทักษะของ ...
✅ Scraped ID 691950: เสด็จสู่สวรรคาลัย "ราชินีแห่งร...
✅ Scraped ID 697743: " เปลี่ยนสินค้าธรรมดา ให้คนตาม...
✅ Scraped ID 542641: สู่จุดสูงสุดของชีวิตด้วย "พีระ...
✅ Scraped ID 697305: "เกมภายใน จิตใจ เทนนิส The Inn...


In [18]:
test_df = pd.DataFrame(test_results)
test_df[['Title','product_type','author','publisher','category_lv1', 'category_lv2']]

,Title,product_type,author,publisher,category_lv1,category_lv2
0,"INPUT - OUTPUT สุดยอดทักษะของ ""คนเก่งงาน""Books...",หนังสือ,คิยามะ ฮิโรทสึงุ,อมรินทร์ How to,จิตวิทยา การพัฒนาตัวเอง,การพัฒนาตัวเอง how to
1,"เสด็จสู่สวรรคาลัย ""ราชินีแห่งราชัน"" คู่พระบารม...",หนังสือ,พินิจ จันทร และคณะ,บันทึกสยาม,หนังสือพระราชนิพนธ์,หนังสือพระราชประวัติราชวงศ์
2,""" เปลี่ยนสินค้าธรรมดา ให้คนตามหาจนต้องจองคิว ผ...",หนังสือ,จิรัฐภณภพ ยอดบริบูรณ์,ดี สนพ./D Publishing,บริหาร ธุรกิจ,การตลาดออนไลน์
3,"สู่จุดสูงสุดของชีวิตด้วย ""พีระมิดสามสุข"" Books...",หนังสือ,ชิอน คาบาซาวะ,อมรินทร์ How to,จิตวิทยา การพัฒนาตัวเอง,การพัฒนาตัวเอง how to
4,"""เกมภายใน จิตใจ เทนนิส The Inner Game of Tenni...",หนังสือ,W. Timothy Gallwey,Dimension,จิตวิทยา การพัฒนาตัวเอง,การพัฒนาตัวเอง how to
